In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [ ]:
vocab_size=10000
#keep only the top 10000 most frequent words
(x_train,y_train),(x_test,y_test)=imdb.load_data(
    num_words=vocab_size
)
print("Training samples:",len(x_train))
print("Testing samples:",len(x_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training samples: 25000
Testing samples: 25000


In [ ]:
max_length=200
x_train=pad_sequences(x_train,maxlen=max_length,padding='post')
x_test=pad_sequences(x_test,maxlen=max_length,padding='post')

In [ ]:
model=Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
   ),
    SimpleRNN(64),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history=model.fit(
    x_train,y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 79ms/step - accuracy: 0.5063 - loss: 0.6960 - val_accuracy: 0.5200 - val_loss: 0.6918
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step - accuracy: 0.5276 - loss: 0.6877 - val_accuracy: 0.5296 - val_loss: 0.6843
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 77ms/step - accuracy: 0.5954 - loss: 0.6486 - val_accuracy: 0.5608 - val_loss: 0.6648
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 78ms/step - accuracy: 0.6357 - loss: 0.5856 - val_accuracy: 0.5884 - val_loss: 0.6509
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 78ms/step - accuracy: 0.6708 - loss: 0.5142 - val_accuracy: 0.6264 - val_loss: 0.6323


In [ ]:
loss,accuracy=model.evaluate(x_test,y_test)
print("Accuracy:",accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.6126 - loss: 0.6346
Accuracy: 0.6126400232315063


In [ ]:
from tensorflow.keras.datasets import imdb
word_index=imdb.get_word_index()
#shift indices by 3 because keras reserves 0,1,2
word_index={k:(v+3) for k,v in word_index.items()}
#add special tokens
word_index['<PAD>']=0
word_index['<START>']=1
word_index['<UNK>']=2
word_index['<UNUSED>']=3

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
def review_to_sequence(review):
  review=review.lower().split()
  sequence=[]
  for word in review:
      sequence.append(word_index.get(word,2))#2=unknown word
  return sequence

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
max_length=200
def predict_sentiment(review):
  sequence=review_to_sequence(review)
  padded=pad_sequences([sequence],maxlen=max_length,padding='post')
  prediction=model.predict(padded,verbose=0)
  score= prediction[0][0]
  print("Sentiment Score:",score)
  if score>0.5:
    print('Positive Review')
  else:
    print('Negative Review')

In [ ]:
predict_sentiment('movie was very good')

Sentiment Score: 0.49372327
Negative Review
